# Assessment 2 — Move AusVille Urban Mobility Analysis
**Unit:** Introduction to Data Science  
**Dataset:** Monthly Average Patronage by Day Type and Mode

---
## Section 1: Data Preparation

### Step 1 — Import libraries

Before we do anything, we need to load the tools (called **libraries**) that Python uses for data work.

- **pandas** (`pd`) — the main library for working with tables of data. Think of it like Excel but in Python.
- **numpy** (`np`) — used for maths and numbers behind the scenes.

The two `pd.set_option` lines just make the output look cleaner — showing all columns and rounding decimals to 2 places.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

### Step 2 — Load the dataset

Here we tell Python where the CSV file is stored and read it into a **DataFrame** (a table with rows and columns).

We call it `df_raw` — the "raw" version before any cleaning. This is good practice so we always have the original to go back to if something goes wrong.

**What to check in the output:** the shape tells you how many rows and columns loaded. We expect 4,300 rows and 7 columns.

In [2]:
file_path = "/Users/apple/INTRO TO DATASCIENCE/Data/Dataset- Monthly average patronage by day type and by mode.csv"

df_raw = pd.read_csv(file_path)

print("Dataset loaded successfully")
print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
df_raw.head()

Dataset loaded successfully
Shape: 4300 rows x 7 columns


,Year,Month,Month_name,Day_of_week,Day_type,Mode,Pax_daily
0,2018,1,Jan,Friday,School Holiday Weekday,MetroBus,268350
1,2018,1,Jan,Friday,School Holiday Weekday,MetroTrain,570700
2,2018,1,Jan,Friday,School Holiday Weekday,RegionalBus,21550
3,2018,1,Jan,Friday,School Holiday Weekday,RegionalTrain,43700
4,2018,1,Jan,Friday,School Holiday Weekday,Tram,529250


### Step 3 — Inspect the structure

Before cleaning anything, we look at what we have. This is like opening a new spreadsheet and checking it makes sense.

- `.dtypes` — shows the **data type** of each column (number, text, etc.). We want Year, Month, and Pax_daily to be numbers, not text.
- `.describe()` — gives summary statistics (min, max, average, etc.) for every column.
- `.info()` — shows column names, types, and how many non-null (non-missing) values there are.

**What to look for:** any columns with unexpected types, or counts less than 4,300 which would mean missing values.

In [3]:
print("Columns:", df_raw.columns.tolist())
print()
print(df_raw.dtypes)
print()
df_raw.describe(include="all")

Columns: ['Year', 'Month', 'Month_name', 'Day_of_week', 'Day_type', 'Mode', 'Pax_daily']

Year           int64
Month          int64
Month_name       str
Day_of_week      str
Day_type         str
Mode             str
Pax_daily      int64
dtype: object



,Year,Month,Month_name,Day_of_week,Day_type,Mode,Pax_daily
count,4300.00,4300.00,4300,4300,4300,4300,4300.00
unique,NaN,NaN,12,7,3,5,NaN
top,NaN,NaN,Apr,Thursday,Normal Weekday,MetroBus,NaN
freq,NaN,NaN,480,680,2265,860,NaN
mean,2021.44,6.55,NaN,NaN,NaN,NaN,240732.31
std,2.27,3.38,NaN,NaN,NaN,NaN,226797.34
min,2018.00,1.00,NaN,NaN,NaN,NaN,1900.00
25%,2019.00,4.00,NaN,NaN,NaN,NaN,37800.00
50%,2021.00,7.00,NaN,NaN,NaN,NaN,165375.00
75%,2023.00,9.00,NaN,NaN,NaN,NaN,406562.50


In [4]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 4300 entries, 0 to 4299
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Year         4300 non-null   int64
 1   Month        4300 non-null   int64
 2   Month_name   4300 non-null   str  
 3   Day_of_week  4300 non-null   str  
 4   Day_type     4300 non-null   str  
 5   Mode         4300 non-null   str  
 6   Pax_daily    4300 non-null   int64
dtypes: int64(3), str(4)
memory usage: 235.3 KB


### Step 4 — Check for data quality issues

Now we specifically look for common problems in the data:

1. **Missing values** — empty cells that could break our analysis.
2. **Duplicate rows** — the same row recorded twice, which would skew averages.
3. **Unexpected category values** — e.g. if `Mode` had a typo like `"MetroBuss"`, it would appear as a 6th mode instead of being grouped with `"MetroBus"`.

**What to look for:** all missing value counts should be 0, duplicates should be 0, and the unique values for each category should look correct with no typos or extra spaces.

In [5]:
print("Missing values:")
print(df_raw.isnull().sum())
print()
print(f"Duplicate rows: {df_raw.duplicated().sum()}")
print()
for col in ["Day_type", "Mode", "Day_of_week"]:
    print(f"Unique {col}: {df_raw[col].unique().tolist()}")
    print()

Missing values:
Year           0
Month          0
Month_name     0
Day_of_week    0
Day_type       0
Mode           0
Pax_daily      0
dtype: int64

Duplicate rows: 0

Unique Day_type: ['School Holiday Weekday', 'Normal Weekday', 'Weekend']

Unique Mode: ['MetroBus', 'MetroTrain', 'RegionalBus', 'RegionalTrain', 'Tram']

Unique Day_of_week: ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']



### Step 5 — Clean and convert data types

Now we fix any issues found in Step 4. We work on a **copy** of the raw data (`df = df_raw.copy()`) so the original is never changed.

- **`.str.strip()`** — removes any invisible leading/trailing spaces from text columns. A space can cause `"MetroBus"` and `"MetroBus "` to be treated as two different values.
- **`.drop_duplicates()`** — removes any identical repeated rows.
- **`.dropna()`** — removes rows where key columns are empty (missing values we cannot work with).

**What to look for:** the cleaned shape should be the same or very close to 4,300 rows. If many rows were dropped, investigate why.

In [6]:
df = df_raw.copy()

# Strip whitespace from text columns
for col in ["Month_name", "Day_of_week", "Day_type", "Mode"]:
    df[col] = df[col].str.strip()

# Remove duplicates and rows missing key fields
df = df.drop_duplicates()
df = df.dropna(subset=["Year", "Month", "Mode", "Pax_daily"])

print(f"Cleaned shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Cleaned shape: 4300 rows x 7 columns


,Year,Month,Month_name,Day_of_week,Day_type,Mode,Pax_daily
0,2018,1,Jan,Friday,School Holiday Weekday,MetroBus,268350
1,2018,1,Jan,Friday,School Holiday Weekday,MetroTrain,570700
2,2018,1,Jan,Friday,School Holiday Weekday,RegionalBus,21550
3,2018,1,Jan,Friday,School Holiday Weekday,RegionalTrain,43700
4,2018,1,Jan,Friday,School Holiday Weekday,Tram,529250


### Step 6 — Feature Engineering

**Feature engineering** means creating new columns from existing ones to make analysis easier or more meaningful.

We create four new columns:

- **`Date`** — combines Year and Month into a proper date (e.g. `2020-03-01`). This lets us plot data on a timeline.
- **`Period_category`** — simplifies `Day_type` into cleaner labels: `"Weekday"`, `"School Holiday"`, or `"Weekend"`. Easier to group and compare.
- **`COVID_period`** — a True/False flag for 2020 and 2021, so we can separate COVID-affected data from normal years.
- **`Network_type`** — groups the 5 transport modes into just `"Metro"` (MetroBus, MetroTrain, Tram) or `"Regional"` (RegionalBus, RegionalTrain).

**What to look for:** the output table should show the new columns populated correctly with no `"Other"` values in `Period_category`.

In [7]:
# Create a Date column from Year and Month
df["Date"] = pd.to_datetime(df[["Year", "Month"]].assign(Day=1))

# Map Day_type into broader period categories
def classify_period(day_type):
    if "School Holiday" in str(day_type):
        return "School Holiday"
    elif "Weekday" in str(day_type) or "Normal" in str(day_type):
        return "Weekday"
    elif "Weekend" in str(day_type):
        return "Weekend"
    return "Other"

df["Period_category"] = df["Day_type"].apply(classify_period)

# Flag COVID-affected years
df["COVID_period"] = df["Year"].isin([2020, 2021])

# Classify Metro vs Regional network
df["Network_type"] = df["Mode"].apply(
    lambda x: "Metro" if "Metro" in str(x) or "Tram" in str(x) else "Regional"
)

print("New features added: Date, Period_category, COVID_period, Network_type")
df[["Year", "Month", "Mode", "Pax_daily", "Date", "Period_category", "COVID_period", "Network_type"]].head(10)

New features added: Date, Period_category, COVID_period, Network_type


,Year,Month,Mode,Pax_daily,Date,Period_category,COVID_period,Network_type
0,2018,1,MetroBus,268350,2018-01-01,School Holiday,False,Metro
1,2018,1,MetroTrain,570700,2018-01-01,School Holiday,False,Metro
2,2018,1,RegionalBus,21550,2018-01-01,School Holiday,False,Regional
3,2018,1,RegionalTrain,43700,2018-01-01,School Holiday,False,Regional
4,2018,1,Tram,529250,2018-01-01,School Holiday,False,Metro
5,2018,1,MetroBus,308450,2018-01-01,Weekday,False,Metro
6,2018,1,MetroTrain,638350,2018-01-01,Weekday,False,Metro
7,2018,1,RegionalBus,22600,2018-01-01,Weekday,False,Regional
8,2018,1,RegionalTrain,47700,2018-01-01,Weekday,False,Regional
9,2018,1,Tram,534800,2018-01-01,Weekday,False,Metro


### Step 7 — Summary statistics

Now the data is clean, we explore it with summary statistics to understand patterns before visualising anything.

- **`.describe()`** — gives the min, max, average, and spread of daily passengers (`Pax_daily`).
- **`groupby("Mode")`** — breaks the data down by transport type so we can compare MetroBus vs MetroTrain vs Tram etc.
- **`groupby("Period_category")`** — compares weekday vs weekend vs school holiday patronage.
- **`groupby("Year")`** — shows the year-on-year trend. We expect a clear dip in 2020-2021 due to COVID.

**What to look for:** which mode has the highest average patronage? Does patronage drop noticeably in 2020-2021?

In [8]:
df["Pax_daily"].describe()

count     4300.00
mean    240732.31
std     226797.34
min       1900.00
25%      37800.00
50%     165375.00
75%     406562.50
max     927650.00
Name: Pax_daily, dtype: float64

In [9]:
df.groupby("Mode")["Pax_daily"].agg(["mean", "median", "std", "min", "max"]).round(0)

,mean,median,std,min,max
Mode,,,,,
MetroBus,281975.00,305975.00,122362.00,22750,503750
MetroTrain,458138.00,458400.00,225300.00,31950,927650
RegionalBus,27079.00,28950.00,11578.00,2350,47200
RegionalTrain,39933.00,41100.00,18942.00,1900,76850
Tram,396537.00,416325.00,178339.00,22750,792550


In [10]:
df.groupby("Period_category")["Pax_daily"].agg(["mean", "median", "std"]).round(0)

,mean,median,std
Period_category,,,
School Holiday,238331.00,195150.00,219508.00
Weekday,281347.00,260050.00,247705.00
Weekend,146640.00,122975.00,137069.00


In [11]:
# Year-on-year average -- shows COVID dip
df.groupby("Year")["Pax_daily"].mean().round(0)

Year
2018   335997.00
2019   334615.00
2020   119098.00
2021   136242.00
2022   209552.00
2023   255160.00
2024   275661.00
2025   271396.00
Name: Pax_daily, dtype: float64

### Step 8 — Export cleaned dataset

We save the cleaned and enriched DataFrame as a new CSV file. This means we do not have to repeat all the cleaning steps every time — we can just load the clean version directly in future notebooks.

The original raw file is left untouched.

In [12]:
output_path = "/Users/apple/INTRO TO DATASCIENCE/Data/ausville_patronage_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to: {output_path}")
print(f"Final shape: {df.shape}")

Cleaned dataset saved to: /Users/apple/INTRO TO DATASCIENCE/Data/ausville_patronage_cleaned.csv
Final shape: (4300, 11)
